# 🐙 Put Octopus online

This sets up your own always-on Octopus on Hugging Face for you. There's nothing to install and nothing to configure by hand.

**You need:** a free Hugging Face account and one token.

1. Make a token: open [huggingface.co/settings/tokens/new?tokenType=write](https://huggingface.co/settings/tokens/new?tokenType=write), type any name (for example `octopus`), press **Create token**, then press **Copy**.
2. Type an invite code below. It's like a password that people need to create an Octopus account (you included).
3. Press ▶ on the cell below and paste the token when it asks.

It takes about a minute here, then Hugging Face builds Octopus for about 10 minutes. You only ever do this once.

In [ ]:
#@title ▶ Put Octopus online
INVITE_CODE = "" #@param {type:"string"}
import subprocess, sys, time, urllib.request
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"], check=True)
from getpass import getpass
from huggingface_hub import HfApi

if len(INVITE_CODE.strip()) < 4:
    raise SystemExit("❌ Type an invite code (4 or more characters) in the INVITE_CODE box above, then press ▶ again.")
token = getpass("Paste your Hugging Face token here and press Enter: ").strip()
api = HfApi(token=token)
try:
    me = api.whoami()
except Exception:
    raise SystemExit("❌ Hugging Face didn't accept that token. Copy it again and press ▶ again.")
role = (me.get("auth", {}).get("accessToken", {}) or {}).get("role")
if role == "read":
    raise SystemExit("❌ That token can only read. Make a new one with the link above (it picks Write for you) and press ▶ again.")
user = me["name"]
space = f"{user}/octopus"
print(f"👋 Hi {user}. Setting up {space}…")

dockerfile = urllib.request.urlopen("https://raw.githubusercontent.com/physicstheory/octopus/main/cloud/Dockerfile").read()
readme = b"""---
title: Octopus
emoji: \xf0\x9f\x90\x99
colorFrom: indigo
colorTo: blue
sdk: docker
app_port: 7860
pinned: false
---

My own AI assistant, Octopus: https://github.com/physicstheory/octopus
"""
try:
    api.create_repo(space, repo_type="space", space_sdk="docker", private=False, exist_ok=True)
    api.add_space_secret(space, "OCTOPUS_PASSWORD", INVITE_CODE.strip())
    api.add_space_secret(space, "HF_TOKEN", token)
    api.upload_file(path_or_fileobj=readme, path_in_repo="README.md", repo_id=space, repo_type="space", commit_message="Octopus")
    api.upload_file(path_or_fileobj=dockerfile, path_in_repo="Dockerfile", repo_id=space, repo_type="space", commit_message="Octopus")
except Exception as exc:
    raise SystemExit(f"❌ Hugging Face said: {exc}\nIf it mentions permissions, make a new token with the Write link above and press ▶ again.")

url = f"https://{user.lower().replace('_', '-').replace('.', '-')}-octopus.hf.space"
print("✅ Done. Hugging Face is now building Octopus (about 10 minutes, only this once).")
print(f"   Watch it here: https://huggingface.co/spaces/{space}")
print(f"   Your Octopus will be at: {url}")
print("   When it opens, create your account with your invite code, then bookmark the page.\n")
last = None
for _ in range(120):  # keep an eye on it for up to 30 minutes
    try:
        stage = api.get_space_runtime(space).stage
    except Exception:
        stage = None
    if stage != last:
        print(time.strftime("%H:%M"), {"BUILDING": "Building…", "RUNNING_BUILDING": "Building…", "APP_STARTING": "Starting…",
              "RUNNING": "🎉 Octopus is online! Open " + url, "BUILD_ERROR": "❌ The build failed. Open the Space link above and look at the Logs.",
              "RUNTIME_ERROR": "❌ Octopus stopped. Open the Space link above and look at the Logs."}.get(stage, stage))
        last = stage
    if stage in ("RUNNING", "BUILD_ERROR", "RUNTIME_ERROR"):
        break
    time.sleep(15)